In [1]:
import flax.nnx as nnx
import orbax.checkpoint as ocp
import jax.numpy as jnp
import jax
import optax
from datasets import Dataset


In [4]:
class SomeModel(nnx.Module):
    def __init__(self, *, rngs):
        self.l1 = nnx.Linear(in_features=2, out_features=2, rngs=rngs)
        self.l2 = nnx.Linear(in_features=2, out_features=1, rngs=rngs)
    def __call__(self, x):
        x = self.l1(x)
        x = nnx.relu(x)
        x = self.l2(x)
        return x


rngs = nnx.Rngs(42)
model = SomeModel(rngs=rngs)
model(jnp.array([[0.1, 0.2]]))


Array([[0.08420833]], dtype=float32)

In [5]:
dataset_length=10_000
source = jax.random.uniform(minval=-1.0, maxval=1.0, shape=(dataset_length,2), key=jax.random.key(43))
dataset = Dataset.from_dict(
    dict(x=source, y=jnp.mean(source, axis=-1, keepdims=True) + jnp.max(source, axis=-1, keepdims=True))
).with_format("jax")

head = dataset[0:5]
head, model(head["x"]), head["y"]


({'x': Array([[ 0.05994868,  0.13650298],
         [ 0.59970164, -0.44060016],
         [-0.08789301, -0.8711016 ],
         [ 0.75174475,  0.204139  ],
         [-0.03630185, -0.27988315]], dtype=float32),
  'y': Array([[ 0.23472881],
         [ 0.6792524 ],
         [-0.5673903 ],
         [ 1.2296866 ],
         [-0.19439435]], dtype=float32)},
 Array([[0.05869697],
        [0.        ],
        [0.        ],
        [0.07915352],
        [0.        ]], dtype=float32),
 Array([[ 0.23472881],
        [ 0.6792524 ],
        [-0.5673903 ],
        [ 1.2296866 ],
        [-0.19439435]], dtype=float32))

In [6]:
def loss_fn(model, x, gt):
    pred = model(x)
    return jnp.mean((pred - gt) ** 2)


loss_fn(model, head["x"], head["y"])

Array(0.43516365, dtype=float32)

In [7]:
@nnx.jit
def train_step(model, optimizer, x, gt):
    loss, grads = nnx.value_and_grad(loss_fn)(model, x, gt)
    optimizer.update(model, grads)
    return loss

optimizer = nnx.Optimizer(model=model, tx=optax.adamw(learning_rate=1e-3), wrt=nnx.Param)
train_step(model, optimizer, head["x"], head["y"])
    

Array(0.43516365, dtype=float32)

In [8]:
_, state = nnx.split(model)
state

State({
  'l1': {
    'bias': Param( # 2 (8 B)
      value=Array([0.00099999, 0.00099999], dtype=float32)
    ),
    'kernel': Param( # 4 (16 B)
      value=Array([[ 0.0592523 , -0.37080714],
             [ 0.95876956,  0.398026  ]], dtype=float32)
    )
  },
  'l2': {
    'bias': Param( # 1 (4 B)
      value=Array([0.00099999], dtype=float32)
    ),
    'kernel': Param( # 2 (8 B)
      value=Array([[0.3317586 ],
             [0.44916335]], dtype=float32)
    )
  }
})

In [12]:
_, optimizer_state = nnx.split(optimizer)
optimizer_state

State({
  'opt_state': {
    0: {
      'count': OptArray( # 1 (4 B)
        value=Array(781, dtype=int32)
      ),
      'mu': {
        'l1': {
          'bias': OptVariable( # 2 (8 B)
            value=Array([-0.05595834, -0.03492408], dtype=float32)
          ),
          'kernel': OptVariable( # 4 (16 B)
            value=Array([[-0.03088722, -0.04012504],
                   [ 0.03347103,  0.02533993]], dtype=float32)
          )
        },
        'l2': {
          'bias': OptVariable( # 1 (4 B)
            value=Array([0.17192848], dtype=float32)
          ),
          'kernel': OptVariable( # 2 (8 B)
            value=Array([[-0.03642246],
                   [-0.01339434]], dtype=float32)
          )
        }
      },
      'nu': {
        'l1': {
          'bias': OptVariable( # 2 (8 B)
            value=Array([0.00733796, 0.00179972], dtype=float32)
          ),
          'kernel': OptVariable( # 4 (16 B)
            value=Array([[0.00861005, 0.00893571],
                   

In [11]:
epochs = 5
batch_size=64
for epoch in range(epochs):
    ds = dataset.shuffle(epoch)
    for i, batch in enumerate(ds.iter(batch_size, drop_last_batch=True)):
        loss = train_step(model, optimizer, batch["x"], batch["y"])
    print(f"{epoch+1}: {loss:.4f}")

1: 0.5928
2: 0.3472
3: 0.2487
4: 0.1759
5: 0.1641
